In [ ]:
using Bessels
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl")

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 3.8533*2*pi
ω_b_0 = 7.0515*2*pi

Lamb_shift = 0.05 #Lamb shift in GHz


φ_a = 0.1108434
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b_0
E_Cb = C_b_inv/2

E_J = 37.12*2*pi


##Drive


ω_d=ω_b_0#+Lamb_shift*2*pi # Drive frequency
ω_p=2ω_a-ω_d

ϵ_0 = 0.5pi
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10.12*g2

buffer_phase = 0.0
pump_phase = 0.0;

##Env

κ_b=1/10.4; # 1/ns from Leon's paper
κ_2 = 4*g2^2/κ_b
κ_a = 0#3e-3*κ_2 #from Leon's paper

In [ ]:
#ATS + Mean_Field
function equation_test!(ddu, du, u, p::SimulationParams{SystemATS_prepared, D, MeanField}, t) where D<:DriveTwoModes_prepared
    ω_a, ω_b, E_J, Z_a, Z_b, ϕ_quantum = p.system_prepared.system.ω_a, p.system_prepared.system.ω_b, p.system_prepared.system.E_J, p.system_prepared.Z_a, p.system_prepared.Z_b, p.system_prepared.ϕ_quantum
    ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase =  p.drive_prepared.drive.ϵ_d, p.drive_prepared.drive.ω_d, p.drive_prepared.drive.buffer_phase, p.drive_prepared.drive.ϵ_0, p.drive_prepared.drive.ω_p, p.drive_prepared.drive.pump_phase
    ϕ_a = u[1]
    ϕ_b = u[2]

    red_ϕ_a = ϕ_a * sqrt(Z_a)/ϕ_quantum
    red_ϕ_b = ϕ_b * sqrt(Z_b)/ϕ_quantum
    
    coupling_1 = (-1/2*(red_ϕ_a + red_ϕ_b)^2+1/24*red_ϕ_a^4+1/24*red_ϕ_b^4) / ϕ_quantum
    #coupling_2 = red_ϕ_a*red_ϕ_b*1/24*(4*red_ϕ_a^2+4*red_ϕ_b^2+6*red_ϕ_a*red_ϕ_b)/ ϕ_quantum
    coupling_2 = red_ϕ_a*red_ϕ_b*1/24*(4*red_ϕ_a^2)/ ϕ_quantum
    coupling_3 = red_ϕ_a*red_ϕ_b*1/24*(4*red_ϕ_b^2)
    coupling_4 = red_ϕ_a*red_ϕ_b*1/24*(6*red_ϕ_a*red_ϕ_b)
    coupling = coupling_1 + coupling_2 + coupling_3 + coupling_4

    #coupling = (-1/2*(red_ϕ_a + red_ϕ_b)^2+1/24*(red_ϕ_a+red_ϕ_b)^4) / ϕ_quantum 
    #coupling = (Taylor_cos(ϕ_a * sqrt(Z_a)/ϕ_quantum + ϕ_b * sqrt(Z_b)/ϕ_quantum, 5) - 1)/ϕ_quantum

    sin_pump = 2*E_J*sin(p.drive_prepared.drive_pump(ϵ_0, ω_p, t, pump_phase))
    ddu[1] = -ω_a*(ω_a*ϕ_a - sqrt(Z_a) * sin_pump*coupling) - 2 * p.environment_prepared.κ_a * du[1]
    ddu[2] = -ω_b*(ω_b*ϕ_b + p.drive_prepared.drive_buffer(ϵ_d, ω_d, t, buffer_phase)*sqrt(2) - sqrt(Z_b) * sin_pump*coupling) - 2 * p.environment_prepared.κ_b * du[2]
    return(nothing)
end

In [ ]:
t_max=20/κ_b
re_a_0, im_a_0 = 0.01, 0
ϕ_a_0, q_a_0=sqrt(2*Z_a)*re_a_0, sqrt(2/Z_a)*im_a_0
ϕ_b_0, q_b_0=0.0, 0.0

system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
drive = DriveTwoModesJA(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)

ϕ_CL_a, ϕ_CL_b, q_CL_a, q_CL_b, t_list_CL = simulator(t_max, system, drive, environment, equation = equation_test!)

In [ ]:
p1,p2 = plot_mode(ϕ_CL_a, q_CL_a, t_list_CL, ω_a, Z_a)

plot_perso(p1,p2,layout=(1,2),
xlabel=L"\Re(a)", ylabel=L"\Im(a)",
title=["Resonator a" "Resonator a RF"],
legend=false)

In [ ]:
t_max=20/κ_b
re_a_0, im_a_0 = 001, 0
ϕ_a_0, q_a_0=sqrt(2*Z_a)*re_a_0, sqrt(2/Z_a)*im_a_0
ϕ_b_0, q_b_0=0.0, 0.0

system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
drive = DriveTwoModesJA(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)

ϕ_CL_a, ϕ_CL_b, q_CL_a, q_CL_b, t_list_CL = simulator(t_max, system, drive, environment);

In [ ]:
p1,p2 = plot_mode(ϕ_CL_a, q_CL_a, t_list_CL, ω_a, Z_a)

plot_perso(p1,p2,layout=(1,2),
xlabel=L"\Re(a)", ylabel=L"\Im(a)",
title=["Resonator a" "Resonator a RF"],
legend=false)

In [ ]:
Taylor_order = 5
t_min = 0.0
system_prepared = prepare(system)
drive_prepared = prepare(drive)
environment_prepared = prepare(system,environment)
u_dot_0, u_0 = get_init(system_prepared, environment_prepared)
dt = 1 / (50* ω_a) # Time step for the simulation

p = SimulationParams_Taylor(Taylor_order, system_prepared, drive_prepared, environment_prepared)

ϕ_a_T, _, q_a_T, _, t_list = default_solver((u_dot_0, u_0), (t_min, t_max), p, dt, equation = equation_Taylor!)

In [ ]:
p1,p2 = plot_mode(ϕ_a_T, q_a_T, t_list, ω_a, Z_a)

plot_perso(p1,p2,layout=(1,2),
xlabel=L"\Re(a)", ylabel=L"\Im(a)",
title=["Resonator a" "Resonator a RF"],
legend=false)

# Try with equations on the fields $a$ and $b$

In [ ]:
#ATS + Mean_Field + ladder_operator
function equation_test!(du,u,p::SimulationParams{SystemATS_ladder_prepared, D, MeanField}, t) where D<:DriveTwoModes_prepared
    ω_a, ω_b, E_J, Z_a, Z_b, ϕ_quantum = p.system_prepared.system.ω_a, p.system_prepared.system.ω_b, p.system_prepared.system.E_J, p.system_prepared.Z_a, p.system_prepared.Z_b, p.system_prepared.ϕ_quantum
    ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase =  p.drive_prepared.drive.ϵ_d, p.drive_prepared.drive.ω_d, p.drive_prepared.drive.buffer_phase, p.drive_prepared.drive.ϵ_0, p.drive_prepared.drive.ω_p, p.drive_prepared.drive.pump_phase

    a = u[1]
    b = u[2]

    red_a = a * sqrt(Z_a)/ϕ_quantum
    red_b = b * sqrt(Z_b)/ϕ_quantum
    red_a_conj = conj(red_a)
    red_b_conj = conj(red_b)
    red_ϕ_a = (red_a + red_a_conj)/sqrt(2)
    red_ϕ_b = (red_b + red_b_conj)/sqrt(2)

    #coupling_1_a = - (red_ϕ_a^2+ red_a_conj*red_b)
    coupling_1_a = - (red_ϕ_a+ red_ϕ_b)^2
    coupling_2_a = 0/12*(red_ϕ_a^4 + red_a^3*red_b_conj + 3*red_a*red_a_conj^2 + 6*red_a_conj*red_b_conj*red_b^2)
    coupling_3_a = 0/12*(3*red_a*red_a_conj*(red_b+red_b_conj)^2+red_b*(red_a+red_a_conj)^3+3*red_b^2*red_b*(red_a+red_a_conj))
    coupling_4_a = 0/12*(3/2*red_a_conj^2*(red_a^2+red_b^2)+red_a^3*red_a_conj+3*red_a^2*red_b*red_b_conj+6*red_a*red_b_conj*red_a*red_b_conj+3/2*red_b^2*red_b_conj^2)
    coupling_2_a = 1 / 12 * (red_ϕ_a^4 + red_ϕ_b^4 + 4*red_ϕ_a*red_ϕ_b^3 + 6*red_ϕ_a^2*red_ϕ_b^2 + 4*red_a^3*red_b)
    coupling_a = sqrt(Z_a)*(coupling_1_a + coupling_2_a + coupling_3_a + coupling_4_a) / ϕ_quantum / sqrt(2) / 2
    

    #coupling_1_b = - (red_ϕ_b^2 + red_a^2/2)
    coupling_1_b = - (red_ϕ_a+ red_ϕ_b)^2
    coupling_2_b = 0/12*(red_ϕ_b^4 + red_a^3*red_a_conj + 3*red_a^2*red_b*red_b_conj + 3*red_a_conj^2*red_b^2)
    coupling_3_b = 0/12*(3*red_b*red_b_conj*(red_a+red_a_conj)^2+red_a*(red_b+red_b_conj)^3+3*red_a^2*red_a*(red_b+red_b_conj))
    coupling_4_b = 0/12*(red_a^3*red_b_conj + 3*red_a^2*red_a*red_b + 3*red_a*red_a_conj^2*red_b + 3*red_a*red_b^2*red_b_conj+3*red_a_conj*red_b^2*red_b_conj)
    coupling_2_b = 1 / 12 * (red_ϕ_a^4 + red_ϕ_b^4 + 4*red_ϕ_a*red_ϕ_b^3 + 6*red_ϕ_a^2*red_ϕ_b^2 + 4*red_a^3*red_b)
    coupling_b = sqrt(Z_b)*(coupling_1_b + coupling_2_b + coupling_3_b + coupling_4_b) / ϕ_quantum / sqrt(2) / 2

    sin_pump = -2*E_J*sin(p.drive_prepared.drive_pump(ϵ_0, ω_p, t, pump_phase))
        
    du[1] = -1im*(ω_a*a + sin_pump*coupling_a) - p.environment_prepared.κ_a * a
    du[2] = -1im*(ω_b*b + p.drive_prepared.drive_buffer(ϵ_d, ω_d, t, buffer_phase) + sin_pump*coupling_b) - p.environment_prepared.κ_b * b
    return nothing
end

In [ ]:
t_max=20/κ_b
a_0 = 1
b_0 = 0.0

system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0, b_0)
drive = DriveTwoModesJA(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)
a, b, t_list = simulator(t_max, system, drive, environment, equation = equation_test!)
a_rot = a.*exp.(1im*ω_a*t_list);

In [ ]:
plot(real.(a_rot[1:end]), imag.(a_rot[1:end]))

In [ ]:
t_max=20/κ_b
a_0 = 0.0
b_0 = 0.0

system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0, b_0)
drive = DriveTwoModesJA(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)
a, b, t_list = simulator(t_max, system, drive, environment, equation = equation_test!)
a_rot_bis = a.*exp.(1im*ω_a*t_list);

In [ ]:
#ATS + Mean_Field + ladder_operator
function equation_test!(du,u,p::SimulationParams{SystemATS_ladder_prepared, D, MeanField}, t) where D<:DriveTwoModes_prepared
    ω_a, ω_b, E_J, Z_a, Z_b, ϕ_quantum = p.system_prepared.system.ω_a, p.system_prepared.system.ω_b, p.system_prepared.system.E_J, p.system_prepared.Z_a, p.system_prepared.Z_b, p.system_prepared.ϕ_quantum
    ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase =  p.drive_prepared.drive.ϵ_d, p.drive_prepared.drive.ω_d, p.drive_prepared.drive.buffer_phase, p.drive_prepared.drive.ϵ_0, p.drive_prepared.drive.ω_p, p.drive_prepared.drive.pump_phase

    a = u[1]
    b = u[2]

    red_a = a * sqrt(Z_a)/ϕ_quantum
    red_b = b * sqrt(Z_b)/ϕ_quantum
    red_a_conj = conj(red_a)
    red_b_conj = conj(red_b)
    red_ϕ_a = (red_a + red_a_conj)/sqrt(2)
    red_ϕ_b = (red_b + red_b_conj)/sqrt(2)

    coupling_1 = - (red_ϕ_a+ red_ϕ_b)^2 + 1 / 12 * (red_ϕ_a^4 + red_ϕ_b^4)
    coupling_2_a = red_ϕ_a*red_ϕ_b^3 / 3
    coupling_3_a = red_ϕ_a^2*red_ϕ_b^2 / 2
    coupling_4_a = red_a^3*red_b / 3
    coupling_a = sqrt(Z_a)*(coupling_1 + coupling_2_a + coupling_3_a + coupling_4_a) / ϕ_quantum / sqrt(2) / 2    

    coupling_2_b = red_ϕ_a*red_ϕ_b^3 / 3
    coupling_3_b = red_ϕ_a^2*red_ϕ_b^2 / 2
    coupling_4_b = 0#red_ϕ_a^3*red_ϕ_b / 3
    coupling_b = sqrt(Z_b)*(coupling_1 + coupling_2_b + coupling_3_b + coupling_4_b) / ϕ_quantum / sqrt(2) / 2

    sin_pump = -2*E_J*sin(p.drive_prepared.drive_pump(ϵ_0, ω_p, t, pump_phase))
        
    du[1] = -1im*(ω_a*a + sin_pump*coupling_a) - p.environment_prepared.κ_a * a
    du[2] = -1im*(ω_b*b + p.drive_prepared.drive_buffer(ϵ_d, ω_d, t, buffer_phase) + sin_pump*coupling_b) - p.environment_prepared.κ_b * b
    return nothing
end

In [ ]:
using JSON
poly_dict_0 = JSON.parsefile("/Users/tmalas/Desktop/Dev/Simulations_local/Code_python/output.json")
poly_dict = Dict(
    Tuple(parse.(Int64, split(k[2:end-1], ','))) => parse(Float64, v)
    for (k, v) in poly_dict_0
);

In [ ]:
function tuple_to_num(red_a,red_b,red_a_conj,red_b_conj,tuple)
    return(red_a^tuple[1]*red_b^tuple[2]*red_a_conj^tuple[3]*red_b_conj^tuple[4])
end

In [ ]:
keys(poly_dict)

In [ ]:
#ATS + Mean_Field + ladder_operator
function equation_test!(du,u,p::SimulationParams{SystemATS_ladder_prepared, D, MeanField}, t) where D<:DriveTwoModes_prepared
    ω_a, ω_b, E_J, Z_a, Z_b, ϕ_quantum = p.system_prepared.system.ω_a, p.system_prepared.system.ω_b, p.system_prepared.system.E_J, p.system_prepared.Z_a, p.system_prepared.Z_b, p.system_prepared.ϕ_quantum
    ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase =  p.drive_prepared.drive.ϵ_d, p.drive_prepared.drive.ω_d, p.drive_prepared.drive.buffer_phase, p.drive_prepared.drive.ϵ_0, p.drive_prepared.drive.ω_p, p.drive_prepared.drive.pump_phase

    a = u[1]
    b = u[2]

    red_a = a * sqrt(Z_a)/ϕ_quantum
    red_b = b * sqrt(Z_b)/ϕ_quantum
    red_a_conj = conj(red_a)
    red_b_conj = conj(red_b)
    red_ϕ_a = (red_a + red_a_conj)/sqrt(2)
    red_ϕ_b = (red_b + red_b_conj)/sqrt(2)

    coupling_1 = - (red_ϕ_a+ red_ϕ_b)^2
    coupling_2 = sum(tuple_to_num(red_a,red_b,red_a_conj,red_b_conj,key)*value / 12 for (key,value) in poly_dict)
    coupling_a = sqrt(Z_a)*(coupling_1 + coupling_2) / ϕ_quantum / sqrt(2) / 2    

    #coupling_2_b = tuple_to_num(red_a,red_b,red_a_conj,red_b_conj,key)*value / 12
    coupling_b = sqrt(Z_b)*(coupling_1 + coupling_2) / ϕ_quantum / sqrt(2) / 2

    sin_pump = -2*E_J*sin(p.drive_prepared.drive_pump(ϵ_0, ω_p, t, pump_phase))
        
    du[1] = -1im*(ω_a*a + sin_pump*coupling_a) - p.environment_prepared.κ_a * a
    du[2] = -1im*(ω_b*b + p.drive_prepared.drive_buffer(ϵ_d, ω_d, t, buffer_phase) + sin_pump*coupling_b) - p.environment_prepared.κ_b * b
    return nothing
end

In [ ]:
#ATS + Mean_Field + ladder_operator
function equation_test!(du,u,p, t)
    ω_a, ω_b, E_J, Z_a, Z_b, ϕ_quantum = p.system_prepared.system.ω_a, p.system_prepared.system.ω_b, p.system_prepared.system.E_J, p.system_prepared.Z_a, p.system_prepared.Z_b, p.system_prepared.ϕ_quantum
    ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase =  p.drive_prepared.drive.ϵ_d, p.drive_prepared.drive.ω_d, p.drive_prepared.drive.buffer_phase, p.drive_prepared.drive.ϵ_0, p.drive_prepared.drive.ω_p, p.drive_prepared.drive.pump_phase
    poly_dict_bis = p.poly_dict_bis

    a = u[1]
    b = u[2]

    red_a = a * sqrt(Z_a)/ϕ_quantum
    red_b = b * sqrt(Z_b)/ϕ_quantum
    red_a_conj = conj(red_a)
    red_b_conj = conj(red_b)
    red_ϕ_a = (red_a + red_a_conj)/sqrt(2)
    red_ϕ_b = (red_b + red_b_conj)/sqrt(2)

    coupling_1 = - (red_ϕ_a+ red_ϕ_b)^2
    coupling_2 = sum(tuple_to_num(red_a,red_b,red_a_conj,red_b_conj,key)*value / 12 for (key,value) in poly_dict_bis)
    coupling_a = sqrt(Z_a)*(coupling_1 + coupling_2) / ϕ_quantum / sqrt(2) / 2    

    #coupling_2_b = tuple_to_num(red_a,red_b,red_a_conj,red_b_conj,key)*value / 12
    coupling_b = sqrt(Z_b)*(coupling_1 + coupling_2) / ϕ_quantum / sqrt(2) / 2

    sin_pump = -2*E_J*sin(p.drive_prepared.drive_pump(ϵ_0, ω_p, t, pump_phase))
        
    du[1] = -1im*(ω_a*a + sin_pump*coupling_a) - p.environment_prepared.κ_a * a
    du[2] = -1im*(ω_b*b + p.drive_prepared.drive_buffer(ϵ_d, ω_d, t, buffer_phase) + sin_pump*coupling_b) - p.environment_prepared.κ_b * b
    return nothing
end

In [ ]:
length(t_save)

In [ ]:
t_max = 20/κ_b
t_min = 0.0
a_0 = 0.01
b_0 = 0.0

system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0, b_0)
drive = DriveTwoModes(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)
tspan = (t_min, t_max) # Time span for the simulation
#preparation
system_prepared = prepare(system)
drive_prepared = prepare(drive)
environment_prepared = prepare(system,environment)

dt = get_dt(system, environment)
u_0 = get_init(system_prepared, environment_prepared)

poly_change = copy(poly_dict)

for (key,values) in poly_change
    println(key)
    poly_dict_bis = copy(poly_change)   # make a shallow copy
    delete!(poly_dict_bis, key)  
    p = (system_prepared = system_prepared,  drive_prepared = drive_prepared, environment_prepared = environment_prepared, poly_dict_bis = poly_dict_bis)
    prob = ODEProblem(equation_test!, u_0, tspan, p)
    t_save = tspan[1]:dt:tspan[2]
    sol = solve(prob, Tsit5(),dt=dt,
        saveat = t_save
        )
    a = getindex.(sol.u,1)
    t_list = sol.t
    #println(length(t_list))
    if length(t_list) == length(t_save)
        println("OK")
    end
    #a_rot = a.*exp.(1im*ω_a*t_list)
    #display(plot(real.(a_rot[1:end]), imag.(a_rot[1:end])))
end


In [ ]:
t_max = 20/κ_b
t_min = 0.0
a_0 = 0.01
b_0 = 0.0

system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0, b_0)
drive = DriveTwoModes(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)
tspan = (t_min, t_max) # Time span for the simulation
#preparation
system_prepared = prepare(system)
drive_prepared = prepare(drive)
environment_prepared = prepare(system,environment)

dt = get_dt(system, environment)
u_0 = get_init(system_prepared, environment_prepared)

keys_kept = [(0, 2, 1, 1), 
(1, 2, 0, 1),
(1, 1, 2, 0),
(0, 2, 0, 2),
(0, 3, 0, 1),
(2, 1, 0, 1),
(0, 1, 0, 3)
]
poly_dict_bis = Dict(k => poly_dict[k] for k in keys_kept)

p = (system_prepared = system_prepared,  drive_prepared = drive_prepared, environment_prepared = environment_prepared, poly_dict_bis = poly_dict_bis)
prob = ODEProblem(equation_test!, u_0, tspan, p)
t_save = tspan[1]:dt:tspan[2]
sol = solve(prob, Tsit5(),dt=dt,
    saveat = t_save
    )
a = getindex.(sol.u,1)
t_list = sol.t
a_rot = a.*exp.(1im*ω_a*t_list)
plot(real.(a_rot[1:end]), imag.(a_rot[1:end]))


In [ ]:
plot(real.(a_rot[1:end]), imag.(a_rot[1:end]))

In [ ]:
t_max=20/κ_b
a_0 = 0.01
b_0 = 0.0

system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0, b_0)
drive = DriveTwoModesJA(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)
a, b, t_list = simulator(t_max, system, drive, environment)
a_rot = a.*exp.(1im*ω_a*t_list);

In [ ]:
plot(real.(a_rot[1:end]), imag.(a_rot[1:end]))